In [1]:
%%time
%load_ext jupyter_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import wandb
import pandas as pd
from collections import defaultdict


def flatten_dict(d, parent_key="", sep="/"):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)


def prepare_data(data):
    flattened_data = [flatten_dict(item) for item in data]
    return pd.DataFrame(flattened_data)


api = wandb.Api()

# Project is specified by <entity/project-name>
runs = api.runs("openproblems-comp/fast-tbg")

summary_list, config_list, name_list = [], [], []
for run in runs:
    # .summary contains the output keys/values for metrics like accuracy.
    #  We call ._json_dict to omit large files
    summary_list.append(run.summary._json_dict)
    # .config contains the hyperparameters.
    #  We remove special values that start with _.
    config_list.append({k: v for k, v in run.config.items() if not k.startswith("_")})
    # .name is the human-readable name of the run.
    name_list.append(run.name)
df_summary = prepare_data(summary_list)
df_config = prepare_data(config_list)
# df_name = prepare_data(name_list)

df = pd.concat([pd.DataFrame(name_list, columns=["name"]), df_summary, df_config], axis=1)
# runs_df = pd.DataFrame({"summary": summary_list, "config": config_list, "name": name_list})

# runs_df.to_csv("results/results_jan_22.csv")

CPU times: user 8.79 s, sys: 581 ms, total: 9.37 s
Wall time: 2min


In [2]:
df.to_csv("results/results_jan_25.csv")

In [3]:
[s for s in df.keys() if "samples" in s]

['val/generated_samples/_type',
 'val/generated_samples/count',
 'val/generated_samples/filenames',
 'val/generated_samples/format',
 'val/generated_samples/height',
 'val/generated_samples/width',
 'test/generated_samples/_type',
 'test/generated_samples/count',
 'test/generated_samples/filenames',
 'test/generated_samples/format',
 'test/generated_samples/height',
 'test/generated_samples/width',
 'val/num_eval_samples',
 'test/num_eval_samples',
 'val/generated_samples__d_base_0/_type',
 'val/generated_samples__d_base_0/count',
 'val/generated_samples__d_base_0/filenames',
 'val/generated_samples__d_base_0/format',
 'val/generated_samples__d_base_0/height',
 'val/generated_samples__d_base_0/width',
 'val/generated_samples__d_base_1/_type',
 'val/generated_samples__d_base_1/count',
 'val/generated_samples__d_base_1/filenames',
 'val/generated_samples__d_base_1/format',
 'val/generated_samples__d_base_1/height',
 'val/generated_samples__d_base_1/width',
 'val/generated_samples__d_base

In [4]:
df["model/net/_target_"].value_counts()

model/net/_target_
src.models.components.tarflow.TarFlow                                      1592
src.models.components.tbg.egnn_dynamics_ad2_cat.EGNN_dynamics_AD2_cat       456
src.models.components.wrappers.MultiModelWrapper                            300
src.models.components.egnn_dynamics_consistency.EGNNDynamicsConsistency     163
src.models.components.dit.DIT3D                                              58
src.models.components.tbg.egnn_dynamics.EGNN_dynamics                        52
src.models.components.egnn_dynamics_consistency.EGNN_dynamics_AD2_cat        24
src.models.components.augmented_coupling_flow.EquivariantAugmentedFlow       19
src.models.components.egnn_dynamics_consistency.EGNN_dynamics_AD2             2
Name: count, dtype: int64

In [25]:
import math


def filterer(x):
    if isinstance(x, float) and not math.isfinite(x):
        return False
    return "table" in list(x)


filtered_df = df[
    df["model/_target_"].isin(["src.models.flow_matching_module.FlowMatchLitModule"])
    & ~df["val/cropped_energy_w1"].isna()
    # & df["tags"].apply(filterer)
][
    [
        # "tags",
        "model/_target_",
        "model/net/_target_",
        "test/cropped_energy_w1",
        # "val/effective_sample_size",
        "test/effective_sample_size",
        "data/n_particles",
        # "val/rama/torus_wasserstein",
        "test/rama/torus_wasserstein",
        # "model/sampling_config/num_proposal_samples",
        # "model/sampling_config/num_test_proposal_samples",
    ]
].drop(
    columns=["model/_target_"]
)

filtered_df.sort_values("data/n_particles")

,model/net/_target_,test/cropped_energy_w1,test/effective_sample_size,data/n_particles,test/rama/torus_wasserstein
1818,src.models.components.tbg.egnn_dynamics_ad2_ca...,1.448635,0.151696,22.0,1.057533
1819,src.models.components.tbg.egnn_dynamics_ad2_ca...,1.171007,0.139668,22.0,1.013127
1856,src.models.components.tbg.egnn_dynamics_ad2_ca...,NaN,NaN,22.0,NaN
1855,src.models.components.tbg.egnn_dynamics_ad2_ca...,NaN,NaN,22.0,NaN
1974,src.models.components.tbg.egnn_dynamics_ad2_ca...,NaN,NaN,22.0,NaN
1999,src.models.components.dit.DIT3D,1.644112,0.027077,22.0,1.076241
2028,src.models.components.tbg.egnn_dynamics_ad2_ca...,2.186954,0.101376,22.0,1.065202
1930,src.models.components.tbg.egnn_dynamics_ad2_ca...,NaN,NaN,22.0,NaN
2253,src.models.components.dit.DIT3D,1.809076,0.085354,22.0,0.999138
2252,src.models.components.dit.DIT3D,1.022912,0.100693,22.0,1.048168


In [26]:
renamed_df = filtered_df.replace(
    {
        "src.models.components.tbg.egnn_dynamics_ad2_cat.EGNN_dynamics_AD2_cat": "EQ-CFM",
        "src.models.components.dit.DIT3D": "DiT-CFM",
    }
).rename(columns={"model/net/_target_": "Model", "data/n_particles": "n_particles"})

/tmp/ipykernel_904119/2095559492.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  renamed_df = filtered_df.replace(


In [35]:
renamed_df.dropna().groupby(
    ["Model", "n_particles"]
).count()  # .groupby(["n_particles", "Model"])  # .mean()

test/cropped_energy_w1  test/effective_sample_size  \
Model   n_particles                                                       
DiT-CFM 22.0                              3                           3   
        33.0                              2                           2   
        42.0                              1                           1   
EQ-CFM  22.0                              3                           3   
        33.0                              3                           3   
        42.0                              4                           4   

                     test/rama/torus_wasserstein  
Model   n_particles                               
DiT-CFM 22.0                                   3  
        33.0                                   2  
        42.0                                   1  
EQ-CFM  22.0                                   3  
        33.0                                   3  
        42.0                                   4

In [37]:
renamed_df.dropna().groupby(["Model", "n_particles"]).head(3).groupby(
    ["Model", "n_particles"]
).count()

test/cropped_energy_w1  test/effective_sample_size  \
Model   n_particles                                                       
DiT-CFM 22.0                              3                           3   
        33.0                              2                           2   
        42.0                              1                           1   
EQ-CFM  22.0                              3                           3   
        33.0                              3                           3   
        42.0                              3                           3   

                     test/rama/torus_wasserstein  
Model   n_particles                               
DiT-CFM 22.0                                   3  
        33.0                                   2  
        42.0                                   1  
EQ-CFM  22.0                                   3  
        33.0                                   3  
        42.0                                   3

In [34]:
final_df = (
    renamed_df.dropna()
    .groupby(["Model", "n_particles"])
    .head(3)
    .groupby(
        [
            "n_particles",
            "Model",
        ]
    )
    .mean()
).round(3)
final_df

test/cropped_energy_w1  test/effective_sample_size  \
n_particles Model                                                         
22.0        DiT-CFM                   1.492                       0.071   
            EQ-CFM                    1.602                       0.131   
33.0        DiT-CFM                   0.955                       0.028   
            EQ-CFM                  852.709                       0.073   
42.0        DiT-CFM                   0.500                       0.008   
            EQ-CFM                   13.716                       0.044   

                     test/rama/torus_wasserstein  
n_particles Model                                 
22.0        DiT-CFM                        1.041  
            EQ-CFM                         1.045  
33.0        DiT-CFM                        1.231  
            EQ-CFM                         1.979  
42.0        DiT-CFM                        2.147  
            EQ-CFM                         2.686

In [32]:
final_df.index.dtypes

n_particles    float64
Model           object
dtype: object

In [29]:
print(final_df.to_latex(float_format="{:.3f}".format, ))

\begin{tabular}{llrrr}
\toprule
 &  & test/cropped_energy_w1 & test/effective_sample_size & test/rama/torus_wasserstein \\
n_particles & Model &  &  &  \\
\midrule
\multirow[t]{2}{*}{22.000000} & DiT-CFM & 1.492 & 0.071 & 1.041 \\
 & EQ-CFM & 1.602 & 0.131 & 1.045 \\
\cline{1-5}
\multirow[t]{2}{*}{33.000000} & DiT-CFM & 0.955 & 0.028 & 1.231 \\
 & EQ-CFM & 852.709 & 0.073 & 1.979 \\
\cline{1-5}
\multirow[t]{2}{*}{42.000000} & DiT-CFM & 0.500 & 0.008 & 2.147 \\
 & EQ-CFM & 13.716 & 0.044 & 2.686 \\
\cline{1-5}
\bottomrule
\end{tabular}

